In [1]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 8.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 17.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import torch

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, GRPOConfig, GRPOTrainer
from datasets import load_dataset
from transformers import TextStreamer, TrainingArguments, EarlyStoppingCallback
from huggingface_hub import login
from langdetect import DetectorFactory, detect

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
SEED = 42
DATASET = "Ichsan2895/alpaca-gpt4-indonesian"
MODEL_NAME = "hendrimardani/fine-tuned-model-indoensian-2"
REPOSITORY_ID = "hendrimardani/fine-tuned-model-indonesian-GRPO"
CHAT_TEMPLATE = "llama-3.1"

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)

In [5]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


In [6]:
def format_reward_func(completions, **kwargs):
    rewards = []
    
    # Pattern untuk mengecek format sempurna dari awal hingga akhir
    perfect_pattern = r"^<think>[\s\S]*?</think>\s*<answer>[\s\S]*?</answer>$"
    
    for completion in completions:
        score = 0.0
        text = completion.strip()
        
        # Hitung kemunculan tag untuk mendeteksi duplikasi
        count_think_open = text.count("<think>")
        count_think_close = text.count("</think>")
        count_ans_open = text.count("<answer>")
        count_ans_close = text.count("</answer>")
        
        # 1. Penalti tag duplikat / halusinasi (-0.5)
        # Jika salah satu tag muncul lebih dari 1 kali dalam satu respons
        if count_think_open > 1 or count_think_close > 1 or count_ans_open > 1 or count_ans_close > 1:
            rewards.append(-0.5)
            continue
            
        # 2. Reward Pembuka <think> (+0.2)
        if count_think_open == 1:
            score += 0.2
            
        # 3. Reward Penutup </think> (+0.3)
        if count_think_close == 1:
            score += 0.3
            
        # 4. Reward Format Sempurna (+1.0)
        # Wajib diawali <think>, ditutup </think>, dan diakhiri tag <answer>...</answer>
        if re.match(perfect_pattern, text):
            score += 1.0
            
        rewards.append(score)
        
    return rewards

In [7]:
def reasoning_length_reward_func(completions, **kwargs):
    rewards = []
    MIN_WORDS = 50
    MAX_WORDS = 400
    
    for completion in completions:
        # PENTING: Jika teks terpotong (tidak ada tag penutup), langsung beri 0!
        if "</think>" not in completion:
            rewards.append(0.0)
            continue

        # Hitung kata jika tag penutup ada
        match = re.search(r"<think>([\s\S]*?)</think>", completion)
        if match:
            word_count = len(match.group(1).split())
            if MIN_WORDS <= word_count <= MAX_WORDS:
                rewards.append(1.0)
            else:
                rewards.append(0.2)
        else:
            rewards.append(0.0)

    return rewards

In [8]:
def correctness_reward_func(completions, answer, **kwargs):
    rewards = []
    
    for completion, target in zip(completions, answer):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        target_text = str(target).lower()
        
        # Ambil teks setelah </think> (atau seluruh teks jika tidak pakai tag <answer>)
        parts = text.split("</think>", 1)
        pred_text = parts[1].strip().lower() if len(parts) > 1 else text.strip().lower()

        # Ekstrak kata-kata kunci penting dari target (panjang kata > 4 huruf)
        target_words = set(re.findall(r'\b[a-zA-Z]{5,}\b', target_text))
        
        if not target_words or not pred_text:
            rewards.append(0.0)
            continue

        # Hitung berapa banyak kata kunci target yang berhasil dimunculkan model
        matched_words = sum(1 for word in target_words if word in pred_text)
        coverage_ratio = matched_words / len(target_words)

        # Berikan reward proporsional berdasarkan cakupan kata kunci
        if coverage_ratio >= 0.6:   # 60% kata kunci utama muncul
            rewards.append(1.0)
        elif coverage_ratio >= 0.3: # 30% kata kunci utama muncul
            rewards.append(0.4)
        else:
            rewards.append(0.0)

    return rewards

In [9]:
def language_reward_func(completions, **kwargs):
    # Set seed agar hasil deteksi bahasa konsisten
    DetectorFactory.seed = 0
    rewards = []

    for completion in completions:
        # Bersihkan tag agar deteksi fokus pada isi teks
        clean_text = (
            completion.replace("<think>", "")
            .replace("</think>", "")
            .replace("<answer>", "")
            .replace("</answer>", "")
            .strip()
        )

        if not clean_text:
            rewards.append(0.0)
            continue

        try:
            # Deteksi bahasa utama dari hasil generasi
            detected_lang = detect(clean_text)

            # 'id' adalah kode ISO untuk Bahasa Indonesia
            if detected_lang == "id":
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        except Exception:
            # Jika teks berisi karakter khusus/simbol matematika yang gagal dideteksi
            rewards.append(0.0)

    return rewards

In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None
)

==((====))==  Unsloth 2026.11.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Unsloth 2026.11.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [11]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = CHAT_TEMPLATE
)

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing="unsloth"
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [13]:
dataset = load_dataset(DATASET, split="train")
print(f"Jumlah data training: {len(dataset)}")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data training: 49969


In [14]:
dataset[0]

{'Unnamed: 0': 1,
 'input': 'Saranlah slogan untuk kampanye daur ulang\n',
 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}

In [15]:
system_prompt = """
    Anda adalah asisten pemecah masalah yang analitis.
    Untuk setiap pertanyaan, Anda WAJIB berpikir selangkah demi selangkah di dalam tag <think>...</think>,
    lalu memberikan jawaban akhir di dalam tag <answer>...</answer>.
"""

def prepare_grpo_dataset(example):
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Berapa 10 - 4?"},
        {
            "role": "assistant",
            "content": "<think>\n10 - 4 = 6.\n</think>\n<answer>6</answer>",
        },
        {"role": "user", "content": example["input"]},
    ]

    prompt_str = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True
    )
    return {"prompt": prompt_str, "answer": str(example["output"]).strip()}

In [16]:
dataset = dataset.map(prepare_grpo_dataset, remove_columns=dataset.column_names)
raw_dataset = dataset.train_test_split(test_size=0.1, shuffle=True, seed=SEED)

train_dataset = raw_dataset["train"]
eval_dataset = raw_dataset["test"]
train_dataset[0]

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n\n    Anda adalah asisten pemecah masalah yang analitis.\n    Untuk setiap pertanyaan, Anda WAJIB berpikir selangkah demi selangkah di dalam tag <think>...</think>,\n    lalu memberikan jawaban akhir di dalam tag <answer>...</answer>.\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBerapa 10 - 4?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n<think>\n10 - 4 = 6.\n</think>\n<answer>6</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDeskripsikan sebuah karakter dalam buku "Dracula".\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n',
 'answer': 'Salah satu karakter di buku "Dracula" adalah Count Dracula sendiri. Dia adalah karakter utama dan antagonis utama dari novel. Dracula adalah vampir berusia berabad-abad, yang tinggal di kastil yang membusuk di Pegunungan Karpatia di Transylvania. Dia digambarkan se

In [17]:
training_args = GRPOConfig(
    beta=0.04,
    temperature=0.8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=3,
    max_steps=800,                
    learning_rate=2e-4,
    weight_decay=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    output_dir="outputs",
    logging_steps=10,
    num_generations=4,                       # Jumlah sampel per prompt (grup)
    max_prompt_length=512,                  # Panjang maksimum input
    max_completion_length=1024,              # Panjang maksimum jawaban/reasoning
    push_to_hub=True,                        # Aktifkan fitur upload otomatis ke HF
    hub_model_id=REPOSITORY_ID,
    save_strategy="steps",                   # Strategi penyimpanan berdasarkan langkah/steps
    save_steps=50,                           # Lakukan penyimpanan & push SETIAP 50 STEPS
    hub_strategy="checkpoint",
    metric_for_best_model="eval_loss", 
    greater_is_better=False,       # Set False karena kita ingin loss yang makin kecil/rendah
    max_grad_norm=1.0,
    report_to="tensorboard"
)


In [18]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward_func,
        correctness_reward_func,
        language_reward_func
    ],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args
)

try:
    from huggingface_hub import snapshot_download
    repo_dir = snapshot_download(repo_id=REPOSITORY_ID)
    checkpoint_path = os.path.join(repo_dir, "last-checkpoint")
    trainer.train(resume_from_checkpoint=checkpoint_path)
except:
    trainer.train()

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

... (22 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss


In [19]:
model.push_to_hub_merged(
    repo_id=REPOSITORY_ID,
    tokenizer=tokenizer,
    save_method = "merged_16bit",
)

config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


... (21 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:06<00:00, 76.63s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/hendrimardani/fine-tuned-model-indonesian-GRPO`
